## Extract script
Extract needed variables from cimmyt.db for analysis in R script

In [ ]:
import pandas as pd
import sqlite3

# SQL DB connection
PATH_SQL_DB = r'/data/processed/sql_db/cimmyt.db'
PATH_OUTPUT = r'/data/processed/sql_db/output/'

conn = sqlite3.connect(PATH_SQL_DB)

### view all tables in the database
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(tables_query, conn)

columns_query = "PRAGMA table_info('emissions_soc');"
columns = pd.read_sql_query(columns_query, conn)

In [ ]:
tables

## 2 - Reference

In [ ]:
ref_query = "SELECT * FROM table_02_reference_enriched;"
ref_df = pd.read_sql_query(ref_query, conn)

ref_df.columns

## 3 - Agronomic

In [51]:
agro_query = "SELECT * FROM table_03_agronomic_features;"
agro_df = pd.read_sql_query(agro_query, conn)

# agro_df.columns

## 4 - Economic

In [52]:
econ_query = "SELECT * FROM table_04_economic_features;"
econ_df = pd.read_sql_query(econ_query, conn)

# econ_df.columns

## 5 - Water

In [53]:
# wat_query = "SELECT * FROM table_05_b_water_features;"
wat_query = "SELECT * FROM table_05_water_features;"
wat_df = pd.read_sql_query(wat_query, conn)

# wat_df.columns

## 7 - Ag Inputs

In [54]:
inputs_query = "SELECT * FROM table_07_emissions_ag_inputs;"
inputs_df = pd.read_sql_query(inputs_query, conn)

# inputs_df.columns

## 8 - Residues

In [55]:
res_query = "SELECT * FROM table_08_emissions_residues;"
res_df = pd.read_sql_query(res_query, conn)

# res_df.columns

## 9 - SOC

In [56]:
soc_query = "SELECT * FROM table_09_emissions_soc;"
soc_df = pd.read_sql_query(soc_query, conn)

# soc_df.columns

values in literature:
> “West and Post (2002) concluded that a move from conventional tillage systems to zero tillage (both with residue retention) can sequester on average 48 ± 13 g C m−2 yr−1.”([Govaerts et al., 2009, p. 104](zotero://select/library/items/R595D9T2)) ([pdf](zotero://open-pdf/library/items/LINK7STZ?page=9&annotation=MQMH59WL))

>“Averaging out SOC differences in all of the 161 experiments under reduced and zero tillage, there was an increase of 2.1 t C ha−1 over plowing. However, when only those cases that had apparently reached equilibrium were included (all zero tillage vs. conventional tillage comparisons from temperate regions), mean SOC increased by approximately 12 t C ha−1.”([Govaerts et al., 2009, p. 104](zotero://select/library/items/R595D9T2)) ([pdf](zotero://open-pdf/library/items/LINK7STZ?page=9&annotation=57JXX637))

West and post - 48g C m2 / yr → /1M * 3.67 (CO2) *10000 = 1,7616
Govaerts et al., 2009 12tC/ha → *3.67 / 20 → 2,202 t CO2/ha/yr


## 11 - Net GHG

In [57]:
ghg_query = "SELECT * FROM table_11_emissions_net_ghg;"
ghg_df = pd.read_sql_query(ghg_query, conn)

ghg_df.columns

Index(['farmer_id', 'plot_id', 'logbook_id', 'practice_id', 'year', 'area_ha',
       'product_types', 'flag_missing_emissions_syn_app',
       'flag_missing_emissions_syn_prod', 'flag_missing_emissions_syn_tran',
       'flag_missing_emissions_org_app', 'flag_imputed_amount',
       'input_record_status', 'synth_input_prod_tco2eq',
       'synth_input_tran_dist_tco2eq', 'synth_input_app_tco2eq',
       'org_fert_app_tco2eq', 'flag_residue_emissions',
       'maize_residues_returned_tco2eq', 'maize_grain_residues_burnt_tco2eq',
       'fuel_cons_status', 'fuel_cons_tco2eq', 'irrigation_types',
       'irr_equip_tco2eq', 'irr_app_tco2eq', 'flag_soc_emissions',
       'soc_delta_tco2eq', 'albedo_emissions_tco2eq_ha',
       'albedo_emissions_tco2eq', 'flag_net_ghg', 'flag_net_ghg_wfuel',
       'net_ghg_emissions_wfuel_tco2eq', 'net_ghg_emissions_tco2eq',
       'net_ghg_emissions_tco2eq_ha', 'net_ghg_emissions_wfuel_tco2eq_ha'],
      dtype='object')

## 12 - ESI

In [58]:
esi_query = "SELECT * FROM table_12_esi_scores;"
esi_df = pd.read_sql_query(esi_query, conn)


In [59]:
### merge ref_df and esi_df on practice_id, logbook_id and year, only take conservation_agriculture and veg_type from ref_df
esi_df_2 = esi_df.merge(
    ref_df[["practice_id", "logbook_id", "longitude_w", "latitude_n", "plot_type", "state", "agroecological_hub", "conservation_agriculture", "veg_type"]],
    on=["practice_id", "logbook_id"],
    how="left",
#    indicator=True
)

# esi_df_2._merge.value_counts()

### length of esi_df_2 and esi_df
# print("Length of esi_df:", len(esi_df))
# print("Length of esi_df_2:", len(esi_df_2))

In [60]:
esi_df_3 = esi_df_2.merge(
    agro_df[["practice_id", "logbook_id", "crop_count", "maize_grain_yield_t_ha", "maize_forage_yield_t", "maize_grain_yield_t", "maize_forage_yield_t_ha"]],
    on=["practice_id", "logbook_id"],
    how="left",
    # indicator=True
)

# esi_df_3._merge.value_counts()

In [61]:
esi_df_4 = esi_df_3.merge(
    soc_df[["practice_id", "logbook_id", "ca_fao_dummy", "og_tillage",
            "fi", "fi_level", "soc_ref", "climate_class", "soil_code_ipcc", "maize_residues_surface_cover_dm_t_ha",
            "tillage",  "maize_residues_returned_dm_t_ha"]],
    on=["practice_id", "logbook_id"],
    how="left",
    # indicator=True
)

# esi_df_4._merge.value_counts()

In [62]:
esi_df_5 = esi_df_4.merge(
    inputs_df[["practice_id", "logbook_id", "flag_pesticides_use", "pesticide_amount_kg_ha", "synth_fert_nitrogen_kg_ha", "synth_fert_amount_kg_ha", "org_fert_dry_amount_kg_ha"]],
    on=["practice_id", "logbook_id"],
    how="left",
    # indicator=True
)

# esi_df_5._merge.value_counts()

In [63]:
esi_df_6 = esi_df_5.merge(
    wat_df[["practice_id", "logbook_id", "water_consumption_flag", #"water_sources", 
            # "water_cons_hyb_ml_m3_ha", "water_cons_rep_adj_m3_ha", 
            "water_cons_rep_m3", "water_cons_rep_m3_ha", "irrigation_types"]],
    on=["practice_id", "logbook_id"],
    how="left",
    #  indicator=True
)

# esi_df_6._merge.value_counts()

In [64]:
esi_df_7 = esi_df_6.merge(
    econ_df[["practice_id", "logbook_id", "total_revenue_mxn_ha", "net_income_mxn_ha"]],
    on=["practice_id", "logbook_id"],
    how="left",
#   indicator=True
)

# esi_df_7._merge.value_counts()

In [65]:
esi_df_8 = esi_df_7.merge(
    res_df[["practice_id", "logbook_id", "flag_total_pct", "surface_cover_perc_harvest", "residues_returned_perc_harvest", "burned_perc_harvest", "maize_ag_residues_returned_dm_t_ha"]],
    on=["practice_id", "logbook_id"],
    how="left",
#    indicator=True
)

# esi_df_8._merge.value_counts()

In [66]:
esi_df_9 = esi_df_8.merge(
    ghg_df[["practice_id", "logbook_id", "area_ha", 
        #     "fuel_cons_tco2eq", 
        "irr_equip_tco2eq", "irr_app_tco2eq",
            "synth_input_prod_tco2eq", "synth_input_tran_dist_tco2eq",
    "synth_input_app_tco2eq", "org_fert_app_tco2eq", "maize_residues_returned_tco2eq", "maize_grain_residues_burnt_tco2eq", "soc_delta_tco2eq", 
    "net_ghg_emissions_tco2eq_ha", 'flag_net_ghg', 'flag_soc_emissions', 'flag_residue_emissions', 'flag_missing_emissions_syn_app',
       'flag_missing_emissions_syn_prod', 'flag_missing_emissions_syn_tran',
       'flag_missing_emissions_org_app', 'flag_imputed_amount']],
    on=["practice_id", "logbook_id"],
    how="left",
   # indicator=True
)

# esi_df_9._merge.value_counts()
# esi_df_9

### esi_df

In [ ]:
# save esi_df to as feather``
esi_df_9.to_feather(r'/data/processed/sql_db/output/esi_df.feather')